In [ ]:
from lume_cheetah import LUMECheetahModel,CheetahSimulator
from virtual_accelerator.cheetah.transformer import SLACCheetahTransformer
from virtual_accelerator.cheetah.utils import get_mad_control_mapping, get_control_mad_mapping
from virtual_accelerator.cheetah.generator import generate_variables, split_control_and_observable
from cheetah.accelerator import Segment
from cheetah.particles import ParticleBeam
import torch
import os

# Get path to beam distributions
beam_dist = os.environ.get(
    'BEAM_DISTRIBUTION',
    '/sdf/group/ad/sw/machine-learning/Linac-Simulation-Server/simulation_server/beams'
)
# Create Cheetah particle Beam from file
incoming_beam = ParticleBeam.from_openpmd_file(
    path=os.path.join(beam_dist, "impact_inj_output_YAG03.h5"),
    energy=torch.tensor(64e6),
    dtype=torch.float32,
)
incoming_beam.particle_charges = torch.tensor(1.0)

# Get path to lattice files
lcls_lattice = os.environ.get(
    'LCLS_LATTICE',
    '/sdf/group/ad/sw/scm/repos/optics/lcls-lattice/cheetah'
)

# Create lattice from file
segment = Segment.from_lattice_json(
    os.path.join(lcls_lattice, "nc_hxr.json")
)

#Set end destination from full lattice
segment = segment.subcell(end='otr2')

# Define the simulator using lattice and particle beam
simulator = CheetahSimulator(
    segment=segment,
    initial_beam_distribution=incoming_beam,
)

# Create transformer that handles maps get/set calls
mad_to_control_name = get_mad_control_mapping()
control_name_to_mad = get_control_mad_mapping()
control_name_to_cheetah = { k: v.lower() for k,v in control_name_to_mad.items()}


transformer = SLACCheetahTransformer(control_name_to_cheetah=control_name_to_cheetah)

# Get supported control system variables
# and a control system device to cheetah mapping
# for the model
variables = generate_variables(segment,mad_to_control_name)



# Define the controllable and observable variables
control_variables, observable_variables = split_control_and_observable(variables)

# Create model
model = LUMECheetahModel(
    simulator=simulator,
    transformer=transformer,
    control_variables=control_variables,
    observable_variables=observable_variables,
)


In [ ]:
control_name_to_cheetah

## Testing getting and setting control model variables 

In [ ]:
# Get variables objects
vars = list(model.supported_variables.keys())
model.supported_variables

In [ ]:
# Get variable values
model.get(vars)

In [ ]:
# Set controllable variables and verify their values changed
ctrl_vars = list(control_variables.keys())
new_values = dict(zip(ctrl_vars,[1]*len(ctrl_vars)))
model.set(new_values)
model.get(vars)

In [ ]:
# Reset model and verify variables when back to init values
model.reset()
model.get(vars)